# ⚡ Kaggle Web Panel

**RU:** Веб-панель управления машиной Kaggle (как SSH, только в браузере): терминал,
статистика (CPU/RAM/диск/GPU), установка Ollama + zstd, запуск Cloudflare-туннеля
и установка моделей — всё из левого меню.

**EN:** A browser control panel for your Kaggle machine: shared terminal, server stats
(CPU/RAM/disk/GPU), one-click Ollama + zstd install, Cloudflare tunnel manager
and model installer.

### Как запустить / How to start
1. **Settings → Internet → ON** (обязательно, иначе не скачать зависимости и туннель).
   Для LLM включи ускоритель: **Session options → Accelerator → GPU**.
2. Запусти ячейку ниже — она всё сделает сама (клонирование, зависимости, cloudflared, запуск).
3. Открой напечатанный **URL** и введи **пароль**.
4. Дальше: слева «Установить Ollama + zstd» → «Запустить туннель» уже сделан →
   введи модель (например `llama3.2:3b`) → «Установить модель».

### Твой домен (опционально)
Впиши `cf_token` из Cloudflare Zero Trust → Networks → Tunnels в `CONFIG` ниже,
а в дашборде Cloudflare добавь Public hostname → service `http://localhost:7860`.
Без токена панель поднимет быстрый туннель со случайным `*.trycloudflare.com` адресом.

### Важно
- Панель слушает только localhost — доступ **только через туннель** и **только по паролю**.
- Сессия Kaggle живёт ~12 ч (GPU ~9 ч) — потом просто запусти ячейку заново.


In [ ]:
#@title ⚡ Запустить панель / Launch the panel { display-mode: "form" }
CONFIG = {
    "password": "",   # пароль панели; пусто = сгенерировать автоматически
    "cf_token": "",   # Cloudflare tunnel token (для своего домена, необязательно)
    "domain":  "",    # ваш домен для именованного туннеля (необязательно)
    "port": 7860,
    "repo": "https://github.com/Jedafa/kaggle-web.git",
}

import os, re, secrets, shlex, subprocess, sys, time, urllib.request
from pathlib import Path

HOME = Path.home()
BASE = HOME / ".kaggle-panel"
REPO_DIR = HOME / "kaggle-web"
CF_BIN = BASE / "bin" / "cloudflared"
TUNNEL_LOG = BASE / "tunnel.log"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def pgrep(pat):
    # list form (no shell) — a shell wrapper would self-match the pattern
    return subprocess.run(["pgrep", "-f", pat], capture_output=True).returncode == 0

# ---------- 1. repo ----------
if not (REPO_DIR / "server.py").exists():
    print("⏳ cloning panel repo ...")
    r = sh(f"git clone --depth 1 {CONFIG['repo']} {REPO_DIR}")
    if r.returncode != 0:
        print(r.stderr); raise SystemExit("clone failed — включите Internet в настройках")
else:
    sh(f"cd {REPO_DIR} && git pull --ff-only")
print("✅ repo:", REPO_DIR)

# ---------- 2. python deps ----------
PY = shlex.quote(sys.executable)
r = sh(f"{PY} -m pip install -q aiohttp psutil")
if r.returncode != 0:
    r = sh(f"{PY} -m pip install -q --user aiohttp psutil")
if r.returncode != 0:
    print(r.stdout[-800:], r.stderr[-800:])
    raise SystemExit("pip install failed")
print("✅ deps: aiohttp, psutil")

# ---------- 3. cloudflared ----------
if not CF_BIN.exists():
    print("⏳ downloading cloudflared ...")
    CF_BIN.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF_BIN)
    CF_BIN.chmod(0o755)
print("✅ cloudflared:", CF_BIN)

# ---------- 4. password + .env ----------
PASSWORD = CONFIG["password"].strip() or secrets.token_urlsafe(9)
PORT = int(CONFIG["port"])
(BASE).mkdir(parents=True, exist_ok=True)
(BASE / ".env").write_text(
    "PANEL_PORT=%s\nPANEL_PASSWORD=%s\nCF_TOKEN=%s\nCF_DOMAIN=%s\nTUNNEL_UUID=\n"
    % (shlex.quote(str(PORT)), shlex.quote(PASSWORD),
       shlex.quote(CONFIG["cf_token"].strip()), shlex.quote(CONFIG["domain"].strip())))

# ---------- 5. start panel ----------
subprocess.run(["pkill", "-f", r"python.*server\.py"], capture_output=True)
time.sleep(1)
env = dict(os.environ, PANEL_PASSWORD=PASSWORD, PANEL_PORT=str(PORT))
lf = open(BASE / "panel.log", "ab")
subprocess.Popen([sys.executable, "server.py"], cwd=REPO_DIR, env=env,
                 stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print((BASE / "panel.log").read_text(errors="ignore")[-2000:])
    raise SystemExit("panel did not start")
print(f"✅ panel running on 127.0.0.1:{PORT}")

# ---------- 6. tunnel ----------
if not pgrep("cloudflared tunnel"):
    r = subprocess.run(["bash", str(BASE / "scripts" / "tunnel.sh"), "start"],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).strip() or "[tunnel] no output")

url = None
if not CONFIG["cf_token"].strip():
    print("⏳ waiting for trycloudflare URL ...")
    for _ in range(60):
        try:
            m = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
                           TUNNEL_LOG.read_text(errors="ignore"))
        except OSError:
            m = []
        if m:
            url = m[-1]; break
        time.sleep(1)
    if not url:
        try:
            print("— tunnel.log tail: —")
            print(TUNNEL_LOG.read_text(errors="ignore")[-900:])
        except OSError:
            pass

# ---------- 7. summary ----------
try:
    from IPython.display import HTML, display
except ImportError:  # plain python fallback
    def display(x):
        print(x)
    class HTML(str):
        pass

print()
print("=" * 62)
print("  ⚡ KAGGLE WEB PANEL")
if url:
    print("  🌐  URL:      ", url)
elif CONFIG["cf_token"].strip():
    print("  🌐  URL:      named tunnel — открой домен из Cloudflare дашборда")
else:
    print("  ⚠️  туннель не поднялся — проверь Internet ON и запусти ячейку заново")
print("  🔑  PASSWORD:  ", PASSWORD)
print("=" * 62)

from IPython.display import HTML, display
if url:
    display(HTML(
        f'<div style="padding:14px;border-radius:12px;background:#131722;'
        f'font:600 15px sans-serif;color:#fff">⚡ Панель: '
        f'<a href="{url}" style="color:#8e7bff">{url}</a> &nbsp;|&nbsp; '
        f'пароль: <code style="color:#00cec9">{PASSWORD}</code></div>'))
print("\nℹ️ Сессия Kaggle живёт ~12 ч (GPU ~9 ч). После перезапуска просто запусти ячейку снова — пароль сохранится в .env, но URL (quick-туннель) станет новым.")


---
### Полезное
- Потеряли ссылку? Запустите ячейку ниже.
- Все настройки туннеля (домен/токен) меняются и прямо в панели: кнопка **«Настройки (.env)»**.


In [ ]:
#@title 🔎 Показать URL и пароль ещё раз / Show URL & password again { display-mode: "form" }
import re
from pathlib import Path
try:
    from IPython.display import HTML, display
except ImportError:
    def display(x):
        print(x)
    class HTML(str):
        pass
BASE = Path.home() / ".kaggle-panel"
env = {}
if (BASE / ".env").exists():
    for line in (BASE / ".env").read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, _, v = line.partition("="); env[k.strip()] = v.strip().strip('"').strip("'")
url = None
try:
    m = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
                   (BASE / "tunnel.log").read_text(errors="ignore"))
    url = m[-1] if m else None
except OSError:
    pass
html = f'<div style="padding:14px;border-radius:12px;background:#131722;font:600 15px sans-serif;color:#fff">'
html += "⚡ " + (f'<a href="{url}" style="color:#8e7bff">{url}</a>' if url else "туннель не запущен — запусти в панели «Start tunnel»")
html += f' &nbsp;|&nbsp; пароль: <code style="color:#00cec9">{env.get("PANEL_PASSWORD","?")}</code></div>'
display(HTML(html))
